In [1]:
import torch
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval import models
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# Stage 2 starts small: ONE model, ONE dataset
dataset = "scifact"
model_name = "intfloat/e5-base-v2"

corpus, queries, qrels = GenericDataLoader(data_folder=f"data/{dataset}").load(split="test")
print(f"{dataset}: {len(corpus)} docs, {len(queries)} queries")

f:\Research Paper\AI2ML code-switching benchmark\venv\Lib\site-packages\beir\datasets\data_loader.py:8: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm


Device: cpu


100%|██████████| 5183/5183 [00:00<00:00, 132146.02it/s]

scifact: 5183 docs, 300 queries


In [2]:
beir_model = DRES(models.SentenceBERT(model_name), batch_size=32)
retriever = EvaluateRetrieval(beir_model, score_function="cos_sim")

results = retriever.retrieve(corpus, queries)
print("Retrieval done. Queries scored:", len(results))

Batches: 100%|██████████| 162/162 [10:26<00:00,  3.86s/it]


Retrieval done. Queries scored: 300


In [3]:
ndcg, _map, recall, precision = retriever.evaluate(qrels, results, [10, 100])
mrr = retriever.evaluate_custom(qrels, results, [10], metric="mrr")

print("nDCG@10  :", ndcg["NDCG@10"])
print("Recall@10:", recall["Recall@10"])
print("Recall@100:", recall["Recall@100"])
print("MRR@10   :", mrr["MRR@10"])

nDCG@10  : 0.71047
Recall@10: 0.84639
Recall@100: 0.956
MRR@10   : 0.67362


In [4]:
import os, pandas as pd
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval import models
from beir.retrieval.evaluation import EvaluateRetrieval
from beir.retrieval.search.dense import DenseRetrievalExactSearch as DRES

os.makedirs("results", exist_ok=True)

datasets = ["scifact", "nfcorpus"]
model_list = [
    "intfloat/e5-base-v2",            # English-centric
    "BAAI/bge-base-en-v1.5",          # English-centric
    "intfloat/multilingual-e5-base",  # multilingual
    "BAAI/bge-m3",                    # multilingual
]

rows = []
for ds in datasets:
    corpus, queries, qrels = GenericDataLoader(data_folder=f"data/{ds}").load(split="test")
    for m in model_list:
        print(f"\n>>> Running {m} on {ds} ...")
        beir_model = DRES(models.SentenceBERT(m), batch_size=32)
        retriever = EvaluateRetrieval(beir_model, score_function="cos_sim")
        results = retriever.retrieve(corpus, queries)
        ndcg, _map, recall, precision = retriever.evaluate(qrels, results, [10, 100])
        mrr = retriever.evaluate_custom(qrels, results, [10], metric="mrr")
        rows.append({
            "dataset": ds, "model": m, "condition": "EN",
            "nDCG@10": ndcg["NDCG@10"], "Recall@10": recall["Recall@10"],
            "Recall@100": recall["Recall@100"], "MRR@10": mrr["MRR@10"],
        })
        print(f"    nDCG@10 = {ndcg['NDCG@10']:.4f}")

df = pd.DataFrame(rows)
df.to_csv("results/baseline.csv", index=False)
print("\nSaved -> results/baseline.csv")
df

100%|██████████| 5183/5183 [00:00<00:00, 175945.14it/s]



>>> Running intfloat/e5-base-v2 on scifact ...


Batches: 100%|██████████| 162/162 [10:17<00:00,  3.81s/it]


    nDCG@10 = 0.7105

>>> Running BAAI/bge-base-en-v1.5 on scifact ...


f:\Research Paper\AI2ML code-switching benchmark\venv\Lib\site-packages\huggingface_hub\file_download.py:137: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\SMHC\.cache\huggingface\hub\models--BAAI--bge-base-en-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Batches: 100%|██████████| 162/162 [10:15<00:00,  3.80s/it]


...

Batches: 100%|██████████| 114/114 [07:41<00:00,  4.05s/it]


    nDCG@10 = 0.3113

>>> Running BAAI/bge-m3 on nfcorpus ...


Batches: 100%|██████████| 114/114 [30:04<00:00, 15.83s/it] 


    nDCG@10 = 0.3141

Saved -> results/baseline.csv


,dataset,model,condition,nDCG@10,Recall@10,Recall@100,MRR@10
0,scifact,intfloat/e5-base-v2,EN,0.71047,0.84639,0.95600,0.67362
1,scifact,BAAI/bge-base-en-v1.5,EN,0.73763,0.86589,0.97000,0.70037
2,scifact,intfloat/multilingual-e5-base,EN,0.68874,0.79400,0.93933,0.66221
3,scifact,BAAI/bge-m3,EN,0.64366,0.78344,0.90367,0.60827
4,nfcorpus,intfloat/e5-base-v2,EN,0.35343,0.16908,0.32418,0.56225
5,nfcorpus,BAAI/bge-base-en-v1.5,EN,0.36810,0.17560,0.33200,0.56857
6,nfcorpus,intfloat/multilingual-e5-base,EN,0.31127,0.15020,0.28022,0.49469
7,nfcorpus,BAAI/bge-m3,EN,0.31411,0.14745,0.28259,0.52316
